# WiFiVision — Final Documentation and Visualization

This notebook handles the remaining post-experiment work.

## Included
1. Final architecture diagram (PNG + SVG)
2. Experimental setup summary
3. Per-joint comparison
4. Subject-wise result inspection
5. Statistical summary
6. Report-ready Markdown generation

It uses the verified project results already produced in the previous notebook.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = r"C:\Users\ROHITH KANNA S\WiFiVision"

baseline_dir = os.path.join(ROOT, "data", "processed", "baseline")
graph_dir = os.path.join(ROOT, "data", "processed", "topology_graph")
final_dir = os.path.join(ROOT, "data", "processed", "final_results")

os.makedirs(final_dir, exist_ok=True)

print("Project root:", ROOT)
print("Output directory:", final_dir)

## Final Architecture Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(18, 5))
ax.axis("off")

blocks = [
    ("CSI Window\n(B, 30, 3, 114, 10)", 0.05),
    ("Frame CNN\nSpatial Features", 0.18),
    ("GRU\nTemporal Modeling", 0.31),
    ("Joint Projection\n17 × 32", 0.44),
    ("Topology Graph", 0.57),
    ("GCN × 2", 0.70),
    ("Self-Attention", 0.83),
    ("Pose Head\n17 × 2", 0.96),
]

for i, (label, x) in enumerate(blocks):
    ax.text(
        x, 0.55, label,
        ha="center", va="center", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.55", edgecolor="black", facecolor="white")
    )
    if i < len(blocks) - 1:
        nx = blocks[i + 1][1]
        ax.annotate(
            "", xy=(nx - 0.055, 0.55), xytext=(x + 0.055, 0.55),
            arrowprops=dict(arrowstyle="->")
        )

plt.title("Topology-Aware CSI-Based Human Pose Estimation", fontsize=15)
plt.tight_layout()

png = os.path.join(final_dir, "final_architecture.png")
svg = os.path.join(final_dir, "final_architecture.svg")

plt.savefig(png, dpi=300, bbox_inches="tight")
plt.savefig(svg, format="svg", bbox_inches="tight")
plt.show()

print("Saved:", png)
print("Saved:", svg)

## Load Verified Baseline and Topology Graph Results

In [ ]:
baseline_errors = np.load(
    os.path.join(baseline_dir, "baseline_errors.npy")
).astype(np.float32)

graph_errors = np.load(
    os.path.join(graph_dir, "errors.npy")
).astype(np.float32)

baseline_mpjpe = float(baseline_errors.mean())
graph_mpjpe = float(graph_errors.mean())

improvement = (
    (baseline_mpjpe - graph_mpjpe)
    / baseline_mpjpe
) * 100

print("=" * 70)
print("VERIFIED RESULTS")
print("=" * 70)
print(f"Baseline MPJPE      : {baseline_mpjpe:.6f}")
print(f"Topology Graph MPJPE: {graph_mpjpe:.6f}")
print(f"Improvement         : {improvement:.2f}%")

## Experimental Setup

In [ ]:
setup = pd.DataFrame([
    ["Total sequences", 270],
    ["Training sequences", 189],
    ["Validation sequences", 27],
    ["Test sequences", 54],
    ["Training subjects", "1–7"],
    ["Validation subjects", "8"],
    ["Test subjects", "9–10"],
    ["Window length", 30],
    ["Stride", 15],
    ["CSI input", "(30, 3, 114, 10)"],
    ["Pose target", "(17, 2)"],
    ["Protocol", "Subject-independent"],
], columns=["Setting", "Value"])

print(setup.to_string(index=False))
setup.to_csv(
    os.path.join(final_dir, "experimental_setup.csv"),
    index=False
)

## Per-Joint Comparison

In [ ]:
baseline_per_joint = baseline_errors.mean(axis=0)

graph_per_joint = np.load(
    os.path.join(graph_dir, "per_joint_error.npy")
).astype(np.float32)

per_joint = pd.DataFrame({
    "Joint": np.arange(1, 18),
    "Baseline": baseline_per_joint,
    "Topology Graph": graph_per_joint
})

per_joint["Improvement (%)"] = (
    (per_joint["Baseline"] - per_joint["Topology Graph"])
    / per_joint["Baseline"]
) * 100

print(per_joint.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

per_joint.to_csv(
    os.path.join(final_dir, "per_joint_comparison.csv"),
    index=False
)

In [ ]:
x = np.arange(17)
width = 0.38

plt.figure(figsize=(12, 6))

plt.bar(
    x - width / 2,
    baseline_per_joint,
    width,
    label="Amplitude Baseline"
)

plt.bar(
    x + width / 2,
    graph_per_joint,
    width,
    label="Topology Graph"
)

plt.xlabel("Joint")
plt.ylabel("Mean Joint Error")
plt.title("Per-Joint Error Comparison")
plt.xticks(x, np.arange(1, 18))
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(final_dir, "per_joint_comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(final_dir, "per_joint_comparison.svg"),
    format="svg",
    bbox_inches="tight"
)

plt.show()

## Subject-Wise Generalization

In [ ]:
subject_file = os.path.join(
    graph_dir,
    "subject_wise_results.npz"
)

subject_data = np.load(subject_file)

print("Available fields:", subject_data.files)

for key in subject_data.files:
    print(f"{key}: {subject_data[key]}")

## Statistical Summary

In [ ]:
statistical_summary = pd.DataFrame([
    ["Paired test windows", 972],
    ["Improved windows", 694],
    ["Improved windows (%)", 71.40],
    ["Worsened windows", 278],
    ["Mean error reduction", 0.013924],
    ["95% CI", "[0.012125, 0.015787]"],
    ["Wilcoxon result", "Statistically significant"],
], columns=["Metric", "Value"])

print(statistical_summary.to_string(index=False))

statistical_summary.to_csv(
    os.path.join(final_dir, "statistical_summary.csv"),
    index=False
)

## Generate Report-Ready Methodology and Results

In [ ]:
report = f"""# WiFiVision: CSI-Based Human Pose Estimation

## Experimental Setup

A subject-independent evaluation protocol was used. The dataset contained 270 sequences: 189 for training, 27 for validation, and 54 for testing. Training used subjects 1–7, validation used subject 8, and testing used unseen subjects 9 and 10.

CSI data were segmented using a temporal window of 30 frames and a stride of 15. Each input window had shape `(30, 3, 114, 10)`, while each target consisted of 17 two-dimensional pose keypoints with shape `(17, 2)`.

## Final Architecture

The final model follows:

**CSI Window → Frame CNN → GRU → Joint Projection → Topology Graph → GCN ×2 → Self-Attention → Pose Head → 17×2 Pose**

The CNN extracts spatial CSI features from individual frames. The GRU models temporal dynamics. The temporal representation is projected into joint-specific embeddings. Graph convolution incorporates body-topology relationships, and self-attention models broader dependencies among joints.

## Quantitative Results

The reproducible amplitude baseline achieved an MPJPE of **{baseline_mpjpe:.6f}**.

The Topology Graph model achieved an MPJPE of **{graph_mpjpe:.6f}**.

This represents a relative MPJPE reduction of **{improvement:.2f}%**.

## Generalization

The Topology Graph model was evaluated on unseen subjects 9 and 10. Previously verified subject-wise MPJPE values were approximately 0.138242 and 0.138897, indicating consistent performance across the held-out subjects.

## Statistical Analysis

A paired comparison over 972 test windows showed improvement in 694 windows (71.40%). The mean error reduction was 0.013924, with a 95% bootstrap confidence interval of [0.012125, 0.015787]. A Wilcoxon signed-rank test indicated a statistically significant improvement over the amplitude baseline.

## Conclusion

Among the evaluated models, the Topology Graph model achieved the best verified performance. The results support the use of explicit joint-topology modeling for improving CSI-based human pose estimation.
"""

report_path = os.path.join(
    final_dir,
    "final_methodology_and_results.md"
)

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print("Saved:", report_path)

## Final Deliverable Check

In [ ]:
print("=" * 70)
print("FINAL OUTPUTS")
print("=" * 70)

for filename in sorted(os.listdir(final_dir)):
    print(filename)

print("\nNotebook complete.")